# Moteur de recommandation métier

Cette étape utilise les compétences détectées par similarité sémantique afin de recommander les métiers les plus adaptés au profil utilisateur.

Le moteur s’appuie sur :

- les compétences détectées
- le mapping métier-compétence
- le référentiel métiers
- un score moyen de correspondance

In [1]:
import pandas as pd

In [2]:
jobs_df = pd.read_csv("../data/gold/jobs.csv")
mapping_df = pd.read_csv("../data/gold/mapping.csv")
competencies_df = pd.read_csv("../data/gold/competencies.csv")
user_matches_df = pd.read_csv("../data/gold/user_skill_matches.csv")

In [3]:
print(jobs_df.shape)
print(mapping_df.shape)
print(competencies_df.shape)
print(user_matches_df.shape)

(22, 3)
(385, 2)
(19, 4)
(7, 5)


# Compétences détectées 

In [4]:
detected_skills = user_matches_df[
    ["competency_id", "competency", "similarity_score"]
]

detected_skills

,competency_id,competency,similarity_score
0,C013,Python,0.632241
1,C001,AWS,0.544859
2,C010,Machine Learning,0.522776
3,C018,TensorFlow,0.511118
4,C003,CI/CD,0.474234
5,C014,REST APIs,0.464196
6,C004,Docker,0.451593


# Fusion avec mapping métier-compétence

In [5]:
job_skill_scores = mapping_df.merge(
    detected_skills,
    on="competency_id",
    how="inner"
)

job_skill_scores.head()

,job_id,competency_id,competency,similarity_score
0,J001,C001,AWS,0.544859
1,J001,C010,Machine Learning,0.522776
2,J001,C013,Python,0.632241
3,J002,C004,Docker,0.451593
4,J002,C014,REST APIs,0.464196


# Calcul score par métier

In [6]:
job_scores = job_skill_scores.groupby("job_id").agg(
    matched_skills=("competency_id", "count"),
    recommendation_score=("similarity_score", "mean")
).reset_index()

# Ajouter infos métiers

In [7]:
job_scores = job_scores.merge(
    jobs_df,
    on="job_id",
    how="left"
)

In [8]:
target_jobs = [
    "Senior Data Scientist",
    "Data Scientist",
    "Machine Learning Engineer",
    "DevOps Engineer",
    "Software Engineer",
    "Backend Developer",
    "Full Stack Developer",
    "Data Analyst",
    "Solutions Architect"
]

job_scores = job_scores[
    job_scores["job_title"].isin(target_jobs)
]

# Trier Top 3

In [9]:
top_jobs = job_scores.sort_values(
    by=["recommendation_score", "matched_skills"],
    ascending=False
).head(3)

top_jobs

,job_id,matched_skills,recommendation_score,job_title,category
21,J022,6,0.521131,Data Analyst,Technology
7,J008,6,0.514983,Data Scientist,Technology
14,J015,6,0.514983,DevOps Engineer,Technology


# Ajouter les compétences correspondantes

In [10]:
def get_matched_skills(job_id):
    skills = job_skill_scores[
        job_skill_scores["job_id"] == job_id
    ]["competency"].unique()
    
    return ", ".join(skills)

top_jobs["matched_skill_names"] = top_jobs["job_id"].apply(get_matched_skills)

top_jobs[
    ["job_title", "category", "matched_skills", "recommendation_score", "matched_skill_names"]
]

,job_title,category,matched_skills,recommendation_score,matched_skill_names
21,Data Analyst,Technology,6,0.521131,"Docker, AWS, Machine Learning, REST APIs, Pyth..."
7,Data Scientist,Technology,6,0.514983,"REST APIs, CI/CD, Python, AWS, Machine Learnin..."
14,DevOps Engineer,Technology,6,0.514983,"Python, REST APIs, CI/CD, Machine Learning, AW..."


# Export

In [11]:
top_jobs.to_csv("../data/gold/top_job_recommendations.csv", index=False)

# Conclusion

Le moteur de recommandation identifie les métiers les plus proches du profil utilisateur en croisant les compétences détectées avec le référentiel métier-compétence.

Le système retourne un Top 3 de métiers recommandés avec :

- un score de correspondance
- le nombre de compétences détectées
- les compétences justifiant la recommandation